# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Horisyre/my-flyrank-ml-intern-starter-project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule will assign a score to each individual input based on the distribution of values within the dataset.

For example, to score clicks_30d, I would examine the percentiles(0.25, 0.5, 0.75) of the feature first. Each value would then receive a score from 1 to 4 depending on which range it falls into.

The percentiles for each feature will be predefined using the historical data. This ensures that the scoring criteria remain consistent when the model is tested on new data, rather than changing the scoring thresholds based on the test data.

The same scoring process will then be applied to the other input features. Once each feature has been assigned a score, the individual scores will be summed to produce an overall score for each content item. The resulting scores can then be used to rank the content items according to their overall score.

For example, if the maximum possible combined score is 12 and a content item receives a total score of 9, it would be classified as Moderate.

The resulting classification levels will be(depnding on the number of input features):

    0–4 → Low.
    5–8 → Moderate.
    9-10 → Good.
    11-12 → Excellent.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import os, sys, subprocess,json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/ranking_lifecycle.csv",low_memory=False)

df["content_date"] = df["content_created_at"].apply(
    lambda x: json.loads(x)["value"]
)

df["content_date"] = pd.to_datetime(
    df["content_date"],
    utc=True
)

test_df = df[
    (df["content_date"] >= "2024-11-22") &
    (df["content_date"] < "2026-05-01")
].copy()


clicks_percentiles = {
    "q25": test_df["clicks_30d"].quantile(0.25),
    "q50": test_df["clicks_30d"].quantile(0.50),
    "q75": test_df["clicks_30d"].quantile(0.75)
}
ctr_percentiles  = {
    "q25": test_df["ctr_30d"].quantile(0.25),
    "q50": test_df["ctr_30d"].quantile(0.50),
    "q75": test_df["ctr_30d"].quantile(0.75)
}
health_score_percentiles = {
    "q25": test_df["health_score"].quantile(0.25),
    "q50": test_df["health_score"].quantile(0.50),
    "q75": test_df["health_score"].quantile(0.75)
}


def score(info:dict):
    value = info["value"]
    percentile = info["percentiles"]
    score_value = 1
    if value < percentile["q25"]:
        pass
    elif value > percentile["q25"] and value <= percentile["q50"]:
        score_value = 2
    elif value > percentile["q50"] and value <= percentile["q75"]:
        score_value = 3
    elif value > percentile["q75"]:
        score_value = 4
    return score_value  

def sum(ctr:dict,clicks:dict,health:dict):  
    ctr["percentiles"] = ctr_percentiles
    clicks["percentiles"] = clicks_percentiles
    health["percentiles"] = health_score_percentiles
    ctr_score = score(ctr)
    clicks_score = score(clicks)
    health_score = score(health)
    total = (
    1* ctr_score +
    1* clicks_score +
    1* health_score
    )
    return total

def ranking(ctr:dict,clicks:dict,health:dict):
    sum_value = sum(ctr,clicks,health)
    if sum_value <=4:
        return "low"
    elif sum_value <=8:
        return "moderate"
    elif sum_value <=10:
        return "good"
    else:
        return "excellent"

test_df["evaluation_ranking"] = test_df.apply(
    lambda row: ranking(
        {"value": row["ctr_30d"]},
        {"value": row["clicks_30d"]},
        {"value": row["health_score"]}
    ),
    axis=1
)

y_true = test_df["impression_tier"]
y_pred = test_df["evaluation_ranking"]


accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1)
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

output_df = test_df[["content_hash_id","ctr_30d", "clicks_30d","health_score","evaluation_ranking"]]
output_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)




## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/ranking_lifecycle.csv",low_memory=False)
df["content_date"] = df["content_created_at"].apply(
    lambda x: json.loads(x)["value"]
)

df["content_date"] = pd.to_datetime(
    df["content_date"],
    utc=True
)

test_df = df[
    (df["content_date"] >= "2024-11-22") &
    (df["content_date"] < "2026-05-01")
].copy()

top_20 = (
    test_df[test_df["impression_tier"] == "excellent"]
    .sort_values("impressions_30d", ascending=False)
    .head(20)
)
eval_df = pd.read_csv("work/outputs/baseline_action_score.csv",low_memory=False)

top_20 = top_20.merge(
    eval_df,
    on="content_hash_id",
    how="left"
)
top_20[
    [
        "content_hash_id",
        "impressions_30d",
        "impression_tier",
        "evaluation_ranking"
    ]
]



evaluation_ranking  excellent  good    low  moderate
impression_tier                                     
excellent                1433   508   1233       467
good                     9401  4648   8834      3751
low                     12453  9093  49187     15111
moderate                12811  9530  24244      9266


| Rank | content_hash_id | impressions_30d | Actual Tier | Predicted Ranking | Action | Reason Code | Confidence Note | What Would Make It Wrong? |
|---:|---|---:|---|---|---|---|---|---|
| 1 | content_eadb33b5df496f4a | 531648 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 2 | content_963de14b1f58978f | 524551 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 3 | content_545bb6cc7081ded3 | 500279 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 4 | content_943dc881428182b8 | 340968 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 5 | content_cc26620b2cbb837f | 316430 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 6 | content_21309e9a83c83653 | 268545 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 7 | content_9ef3d7516483e665 | 221019 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 8 | content_33d31496fca9665e | 218829 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 9 | content_acbcc847f8996314 | 216534 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 10 | content_32c5cc913fb4ff41 | 214882 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 11 | content_c60628276389acbb | 198809 | excellent | moderate | Investigate | Significant under-ranking | Low | Current features may be insufficient to identify excellent pages |
| 12 | content_c1f764a2f362d1c3 | 174005 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 13 | content_f107e54b10b43725 | 165296 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 14 | content_d0acf7062bc6b257 | 165234 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 15 | content_62770e1299963fe4 | 162484 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 16 | content_f43118e089ecc69a | 161830 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 17 | content_661a7734f691bef5 | 159829 | excellent | good | Investigate | Baseline under-ranks an excellent page | Medium | Current features may not adequately distinguish good from excellent |
| 18 | content_f352b7cfd0b2f434 | 153577 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 19 | content_ef7013c86d07aa99 | 151404 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |
| 20 | content_471d9cabce329a66 | 149868 | excellent | excellent | Correct classification | Features support excellent ranking | High | Additional unseen factors may affect performance |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The classification results show that one of the weaknesses of the baseline is distinguishing excellent pages from low pages. Of the 3,641 pages that were actually classified as excellent, only 1,433 (39.36%) were correctly classified as excellent. A further 508 (13.95%) were classified as good, while 467 (12.83%) were classified as moderate. Most notably, 1,233 pages (33.86%) were classified as low.

This indicates that the baseline has difficulty distinguishing between excellent and low pages. This may suggest that the model does not have sufficient information to make this distinction, or that the scoring methodology does not adequately capture the characteristics associated with page quality. However in the case of scoring methodology, when weights were applied to scoring of each input I saw the accuracy of the baseline algorithm decrease.

There is no obvious pattern in the top twenty suggesting that the baseline systematically classifies the highest-performing content as low. This may be partly because the analysis focuses specifically on top-performing content and ranks these pages using a secondary feature, impressions_30d. In the top 20 results, the baseline algorithm incorrectly classifies two excellent pages as moderate and good. These are false negatives for the excellent class, as the actual tier is excellent but the baseline assigns a lower ranking. When considering the overall classification results, this may suggest that the baseline is missing important signals needed to distinguish excellent pages from lower-ranking pages. This is particularly noticeable when examining the misclassification of excellent pages as low, while the number of misclassifications between excellent, moderate, and good is comparatively lower. This suggests that the current three-feature rule may have difficulty distinguishing the extremes of the ranking scale, particularly low from excellent.

The features used by the baseline were based on the previous 30 days of historical data and did not incorporate future performance windows. In addition, no product flags were used as inputs in the baseline ranking.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.